# Gate G3 — sample size (antenatal model)

Framework stage: **G3**. Reads `mimiciv.antenatal_predictors_v1` (7-day buffer) and checks with the Riley et al. criteria whether the 129 events are enough for the **27 candidate parameters** of the outcome card. Spanish version: `01_sample_size_gate_es.ipynb` (same code).

**How it is decided.** The framework uses the observed Cox-Snell R² of a cross-validated logistic regression: out-of-fold predictions, 5 folds grouped by patient, imputation of missing values and scaling fitted only on the training part of each fold. Only R² is computed; the AUROC and the coefficients are not looked at in this gate.

**If it does not pass.** Per the framework, the model is first fitted and classified (G4). If it meets the G4 criteria, it can be refitted **once** with the reduced set of 17 parameters fixed in the card. If the data do not support that either, the outcome is reported as a feasibility analysis.

Output (aggregate only): `data/outputs_01_sample_size_gate.txt`.

In [1]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # 'penalty' renamed in recent scikit-learn

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.database import get_connect_database, disconnect_database

SEED = 20261119
BUFFER_DAYS = 7          # main information window
N_FOLDS = 5
S_TARGET = 0.9           # Riley criterion 1: expected shrinkage
DELTA = 0.05             # Riley criterion 2: optimism in Nagelkerke R2
MARGIN = 0.05            # Riley criterion 3: precision of the overall risk
OUT_FILE = ROOT / "data" / "outputs_01_sample_size_gate.txt"

## 1. Data (gate G1: same population and events)

In [2]:
engine = get_connect_database()
df = pd.read_sql(
    "SELECT * FROM mimiciv.antenatal_predictors_v1 WHERE buffer_days = %(b)s",
    engine, params={"b": BUFFER_DAYS})
disconnect_database(engine)

assert len(df) == 19440 and df["hadm_id"].is_unique, "population differs from gate G1"
assert int(df["label_main"].sum()) == 129, "events differ from gate G1"
print(len(df), "admissions,", df["subject_id"].nunique(), "patients,", int(df["label_main"].sum()), "events")

19440 admissions, 13735 patients, 129 events


## 2. Design matrix: the 27 parameters (references: White, Private, Married)

In [3]:
def design_matrix(d):
    """The 27 candidate parameters of the outcome card (section 4). Reference levels:
    White, Private, Married."""
    X = pd.DataFrame(index=d.index)
    X["age"] = d["age"].astype(float)
    for g in ["Black", "Hispanic", "Asian", "Other or unknown"]:
        X[f"race_{g}"] = (d["race_group"] == g).astype(float)
    X["ins_Medicaid"] = (d["insurance_group"] == "Medicaid").astype(float)
    X["ins_Medicare_other_missing"] = (d["insurance_group"] == "Medicare, other or missing").astype(float)
    X["english"] = d["english"].astype(float)
    X["marital_Not_married"] = (d["marital_group"] == "Not married").astype(float)
    X["marital_Missing"] = (d["marital_group"] == "Missing").astype(float)
    for c in ["any_prior_admission", "prior_admission_90d",
              "hx_cesarean", "hx_hypertensive_pregnancy", "hx_chronic_hypertension",
              "hx_pregestational_diabetes", "hx_gestational_diabetes", "hx_substance_use",
              "hx_thyroid_disease", "hx_adverse_pregnancy_outcome", "hx_other_high_risk_chronic"]:
        X[c] = d[c].astype(float)
    X["last_sbp"] = d["last_sbp"].astype(float)
    X["last_dbp"] = d["last_dbp"].astype(float)
    X["any_elevated_bp"] = d["any_elevated_bp"].astype(float)
    X["last_bmi"] = d["last_bmi"].astype(float)
    X["last_hemoglobin"] = d["last_hemoglobin"].astype(float)
    X["had_prenatal_care_here"] = d["had_prenatal_care_here"].astype(float)
    return X

X = design_matrix(df)
y = df["label_main"].astype(int).to_numpy()
groups = df["subject_id"].to_numpy()
P = X.shape[1]
assert P == 27, P
print("candidate parameters:", P)
print("missing values:", X.isna().sum()[X.isna().sum() > 0].to_dict())

candidate parameters: 27
missing values: {'last_sbp': 7524, 'last_dbp': 7524, 'last_bmi': 11136, 'last_hemoglobin': 8005}


## 3. Logistic regression with cross-validation grouped by patient

In [4]:
# Out-of-fold predictions of an unpenalized logistic regression. Imputation and scaling are
# fitted on the training part of each split only; folds are grouped by patient.
def make_model():
    return make_pipeline(
        IterativeImputer(max_iter=10, random_state=SEED),
        StandardScaler(),
        LogisticRegression(penalty=None, max_iter=5000))

oof = np.full(len(y), np.nan)
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (tr, te) in enumerate(cv.split(X, y, groups), 1):
    m = make_model().fit(X.iloc[tr], y[tr])
    oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    print(f"fold {k}: {len(te)} admissions, {y[te].sum()} events")
assert not np.isnan(oof).any()

fold 1: 3910 admissions, 35 events
fold 2: 3876 admissions, 23 events
fold 3: 3865 admissions, 25 events
fold 4: 3905 admissions, 27 events
fold 5: 3884 admissions, 19 events


## 4. Observed Cox-Snell R²

In [5]:
def loglik(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))

n, E = len(y), int(y.sum())
phi = E / n
L0 = n * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))   # intercept-only model
L1 = loglik(y, oof)
R2_cs = 1 - np.exp(2 * (L0 - L1) / n)
R2_max = 1 - np.exp(2 * L0 / n)
print(f"prevalence {phi:.4f}; max Cox-Snell R2 {R2_max:.4f}")
print(f"observed (cross-validated) Cox-Snell R2 {R2_cs:.5f} = {100 * R2_cs / R2_max:.1f}% of the maximum")

prevalence 0.0066; max Cox-Snell R2 0.0767
observed (cross-validated) Cox-Snell R2 -0.00054 = -0.7% of the maximum


## 5. Riley et al. criteria

1. Expected shrinkage of 0.9 or more.
2. Difference of 0.05 or less between apparent and adjusted Nagelkerke R².
3. Overall risk estimated within ±0.05.

`p_supported` is the number of parameters the data support with the observed R².

In [6]:
# Riley et al. (BMJ 2020) criteria for a binary outcome
def riley(p, R2, R2max, phi):
    if R2 <= 0:
        return None
    n1 = p / ((S_TARGET - 1) * np.log(1 - R2 / S_TARGET))
    S2 = R2 / (R2 + DELTA * R2max)
    n2 = p / ((S2 - 1) * np.log(1 - R2 / S2))
    n3 = (1.96 / MARGIN) ** 2 * phi * (1 - phi)
    S = max(S_TARGET, S2)
    p_max = int(np.floor(n * (S - 1) * np.log(1 - R2 / S)))   # parameters the data support
    n_req = int(np.ceil(max(n1, n2, n3)))
    return dict(n1=n1, n2=n2, n3=n3, S2=S2, n_required=n_req,
                events_required=int(np.ceil(n_req * phi)), p_supported=p_max)

res = riley(P, R2_cs, R2_max, phi)
if res is None:
    gate = "NOT PASSED (cross-validated R2 is not positive)"
else:
    gate = "PASSED" if n >= res["n_required"] else "NOT PASSED"
    for k, v in res.items():
        print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")
print(f"available: {n} admissions, {E} events")
print("gate G3 with", P, "parameters:", gate)

available: 19440 admissions, 129 events
gate G3 with 27 parameters: NOT PASSED (cross-validated R2 is not positive)


## 6. Aggregate output

In [7]:
lines = [
    "Gate G3 (sample size), antenatal model; aggregate output only",
    f"buffer_days={BUFFER_DAYS}; folds={N_FOLDS} grouped by patient; seed={SEED}",
    f"admissions={n}; events={E}; prevalence={phi:.4f}; candidate_parameters={P}",
    f"R2_cs_max={R2_max:.4f}; R2_cs_observed_cv={R2_cs:.5f} ({100 * R2_cs / R2_max:.1f}% of max)",
]
if res is not None:
    lines += [f"n_required={res['n_required']} (criterion1={res['n1']:.0f}, criterion2={res['n2']:.0f}, criterion3={res['n3']:.0f}); "
              f"events_required={res['events_required']}; parameters_supported={res['p_supported']}"]
lines += [f"gate={gate}"]
OUT_FILE.parent.mkdir(exist_ok=True)
OUT_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

Gate G3 (sample size), antenatal model; aggregate output only
buffer_days=7; folds=5 grouped by patient; seed=20261119
admissions=19440; events=129; prevalence=0.0066; candidate_parameters=27
R2_cs_max=0.0767; R2_cs_observed_cv=-0.00054 (-0.7% of max)
gate=NOT PASSED (cross-validated R2 is not positive)
